# Лабораторная работа № 2 — создание базы parking

База создаётся только при отсутствии. Существующая база не удаляется. Для административного подключения PGDATABASE должен указывать на postgres либо другую существующую служебную БД.

Выполняйте ячейки сверху вниз (**Run All**). Запросы находятся прямо в ноутбуке; SQL-файлы не читаются. Параметры подключения — в следующей ячейке. Зависимости: `pip install "psycopg[binary]" pandas ipykernel` (также поддерживается psycopg2).

[Все лабораторные](../../README.md) · [Отчёты ЛР2](../../labs/lab02/)

## Содержание

- [1. Создание базы](#part-1)
- [2. Проверка параметров](#part-2)

In [1]:
import os
import getpass
import pandas as pd
from IPython.display import display
try:
    import psycopg
except ImportError:
    import psycopg2 as psycopg

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", None)
# При необходимости задайте PGHOST, PGPORT, PGUSER, PGPASSWORD перед запуском.
# Пароль в ноутбуке не сохраняйте: драйвер также поддерживает ~/.pgpass.
DB_NAME = os.getenv("PGDATABASE", "postgres")
conn = psycopg.connect(
    host=os.getenv("PGHOST", "127.0.0.1"),
    port=int(os.getenv("PGPORT", "5432")),
    user=os.getenv("PGUSER", getpass.getuser()),
    password=os.getenv("PGPASSWORD") or None,
    dbname=DB_NAME,
)
conn.autocommit = True

def q(sql):
    """Выполнить инструкцию; выборку показать таблицей."""
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
            if cur.description:
                frame = pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
                frame.index = range(1, len(frame) + 1)
                display(frame)
                print(f"Строк: {len(frame)}")
            else:
                print(cur.statusmessage)
    except Exception:
        conn.rollback()
        raise

run = q
print("Подключение:", DB_NAME)


Подключение: postgres


<a id="part-1"></a>

## 1. Создание базы

In [2]:
if psycopg.__name__ == "psycopg":
    from psycopg import sql as pgsql
else:
    from psycopg2 import sql as pgsql

TARGET_DATABASE = "parking"
# Локаль наследуется от template0: работает на macOS, Linux и Windows.
# Владелец — текущая роль PostgreSQL; требуется право CREATEDB.
with conn.cursor() as cur:
    cur.execute("SELECT 1 FROM pg_database WHERE datname = %s", (TARGET_DATABASE,))
    if cur.fetchone():
        print("База уже существует:", TARGET_DATABASE)
    else:
        cur.execute(pgsql.SQL("CREATE DATABASE {} ENCODING 'UTF8' TEMPLATE template0").format(
            pgsql.Identifier(TARGET_DATABASE)))
        print("Создана база:", TARGET_DATABASE)


Создана база: parking


<a id="part-2"></a>

## 2. Проверка параметров

In [3]:
q("""
SELECT datname, pg_encoding_to_char(encoding) AS encoding, datcollate, datctype FROM pg_database WHERE datname = 'parking';
""")

,datname,encoding,datcollate,datctype
1,parking,UTF8,C,C


Строк: 1


## Завершение

Выполните эту ячейку, чтобы завершить транзакцию и закрыть подключение. При ошибке изменения текущей транзакции отменяются; повторите ноутбук с начала.

In [4]:
conn.close()